# Interrupt and resume — the HITL primitive

Every HITL pattern in this folder is built on one move: **pause the graph mid-run, persist its state, and continue later with a user-supplied value**. LangGraph exposes this as `interrupt(payload)` inside a node + `Command(resume=value)` to continue. This notebook builds the same surface in pure Python so the moving parts are visible.

The shared scenario for the next six leaves: a research agent that **searches** the corpus, **drafts** an answer, asks the human to **approve** the draft, then **publishes** the citation. Only `approve` calls `interrupt()`.

## What a real LangGraph version looks like

```python
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

class S(TypedDict):
    question: str
    hits: list
    draft: str
    approved: bool
    published: bool

def search(s: S) -> dict:    return {'hits': search_corpus(s['question'])}
def draft(s: S) -> dict:     return {'draft': draft_answer(s['question'], s['hits'])}
def approve(s: S) -> dict:
    decision = interrupt({'reason': 'needs_human_approval', 'draft': s['draft']})
    return {'approved': bool(decision['approved'])}
def publish(s: S) -> dict:   return {'published': s['approved']}

g = StateGraph(S)
for name, fn in [('search', search), ('draft', draft), ('approve', approve), ('publish', publish)]:
    g.add_node(name, fn)
g.add_edge(START, 'search'); g.add_edge('search', 'draft')
g.add_edge('draft', 'approve'); g.add_edge('approve', 'publish'); g.add_edge('publish', END)
app = g.compile(checkpointer=MemorySaver())

thread = {'configurable': {'thread_id': 't1'}}
# 1st call: runs up to the interrupt and returns the payload
first = app.invoke({'question': 'What is RA-MoE?'}, config=thread)
# 2nd call: resume with the human's decision
final = app.invoke(Command(resume={'approved': True}), config=thread)
```

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from hitl import build_research_graph, Checkpointer, Command, get_question

graph = build_research_graph()
cp = Checkpointer()
state = graph.run({'question': get_question('q01'), 'question_id': 'q01'},
                  thread_id='demo', checkpointer=cp)
print('paused at:', state['_interrupt']['node'])
print('reason   :', state['_interrupt']['reason'])
print('draft    :', state['draft'][:120])

Now resume — the human says 'approve'. Notice the resume value is plain JSON; that's what makes this checkpointable across processes.

In [ ]:
state = graph.resume(thread_id='demo',
                     command=Command(resume={'approved': True, 'reviewer': 'demo-user'}),
                     checkpointer=cp)
print('approved :', state['approved'])
print('published:', state['published'])
print('answer   :', state['final_answer'])
print('checkpoints:', len(cp.history('demo')))

## What to persist

The mini-runner's checkpoint is **`(thread_id, step, node, state, interrupt_payload)`**. That's the same shape LangGraph's `MemorySaver` / `PostgresSaver` write — adding the human's decision after the interrupt is what makes resume deterministic.

The remaining leaves in `04-human-in-the-loop/` specialise this primitive: approval gates over tool calls, mid-run state edits, time-travel via fork, streaming with intervention, and pushing the interrupt onto an async queue.